# MongoDB

MongoDB es una base de datos de documentos de propósito general.

## Documentos
Los datos en Mongo se representan como documentos JSON.

Los campos pueden variar entre documentos. 

Se pueden anidar documentos para expresar jerarquías y armar estructuras como arrays.

### Colecciones

Es un grupo de documentos. Como una tabla, pero más flexible: no tienen un schema a menos que lo configures.

### Indices
MongoDB soporta varias estrategias de indexado para soportar ejecución eficiente de queries.

### Pipelines de agregación
MongoDB incorpora un framework para crear pipelines de procesamiento de datos con gran variedad de operadores y expresiones.

In [2]:
from pymongo import MongoClient
import pymongo

uri = "mongodb://mongo:27017/"
client = MongoClient(uri)
client.admin.command("ping")

{'ok': 1.0}

In [14]:
db = client["clase"]

In [12]:
#db["coleccion1"].drop()
#db["coleccion2"].drop()

In [15]:
db.create_collection("coleccion1")
db.create_collection("coleccion2")

Collection(Database(MongoClient(host=['mongo:27017'], document_class=dict, tz_aware=False, connect=True), 'clase'), 'coleccion2')

In [16]:
db.list_collection_names()

['coleccion2', 'coleccion1']

In [17]:
collection = db["coleccion1"]

# Insertar

In [18]:
result = collection.insert_one({"key": "value"})
print(result.acknowledged)

True


In [19]:
document_list = [
   {"key": "value1"},
   {"key": "value2"}
]
result = collection.insert_many(document_list)
print(result.acknowledged)

True


# Actualizar

In [20]:
query_filter = { "key" : "value2" }
update_operation = { "$set" : 
    { "key" : "value" }
}
result = collection.update_one(query_filter, update_operation)
print(result.modified_count)

1


In [21]:
query_filter = { "key" : "value" }
update_operation = { "$set" : 
    { "key" : "value1000" }
}
result = collection.update_many(query_filter, update_operation)
print(result.modified_count)

2


# Reemplazar

In [22]:
query_filter = { "key" : "value1000" }
replace_document = { "another_key" : "another_value" }
result = collection.replace_one(query_filter, replace_document)
print(result.modified_count)

1


# Borrar

In [23]:
query_filter = { "key" : "value1000" }
result = collection.delete_one(query_filter)
print(result.deleted_count)

1


In [24]:
query_filter = { "key" : "value1000" }
result = collection.delete_many(query_filter)
print(result.deleted_count)

0


# Buscar

In [25]:
collection = db["fruits"]
collection.insert_many([
        { "name": "apples", "qty": 5, "rating": 3, "color": "red", "type": ["fuji", "honeycrisp"] },
        { "name": "bananas", "qty": 7, "rating": 4, "color": "yellow", "type": ["cavendish"] },
        { "name": "oranges", "qty": 6, "rating": 2, "type": ["naval", "mandarin"] },
        { "name": "pineapple", "qty": 3, "rating": 5, "color": "yellow" },
])

InsertManyResult([ObjectId('684369047bbf6b41012e9c0b'), ObjectId('684369047bbf6b41012e9c0c'), ObjectId('684369047bbf6b41012e9c0d'), ObjectId('684369047bbf6b41012e9c0e')], acknowledged=True)

In [26]:
results = collection.find({ "color": "yellow" })
list(results)

[{'_id': ObjectId('684369047bbf6b41012e9c0c'),
  'name': 'bananas',
  'qty': 7,
  'rating': 4,
  'color': 'yellow',
  'type': ['cavendish']},
 {'_id': ObjectId('684369047bbf6b41012e9c0e'),
  'name': 'pineapple',
  'qty': 3,
  'rating': 5,
  'color': 'yellow'}]

## Comparación

- `$eq`
- `$gt`
- `$gte`
- `$in`
- `$lt`
- `$lte`
- `$ne`
- `$nin`

[Referencias](https://www.mongodb.com/docs/manual/reference/operator/query-comparison/)

In [27]:
results = collection.find({ "rating": { "$gt" : 2 }})
list(results)

[{'_id': ObjectId('684369047bbf6b41012e9c0b'),
  'name': 'apples',
  'qty': 5,
  'rating': 3,
  'color': 'red',
  'type': ['fuji', 'honeycrisp']},
 {'_id': ObjectId('684369047bbf6b41012e9c0c'),
  'name': 'bananas',
  'qty': 7,
  'rating': 4,
  'color': 'yellow',
  'type': ['cavendish']},
 {'_id': ObjectId('684369047bbf6b41012e9c0e'),
  'name': 'pineapple',
  'qty': 3,
  'rating': 5,
  'color': 'yellow'}]

## Operadores lógicos

- `$and`
- `$not`
- `$nor`
- `$or`

In [28]:
results = collection.find({ 
    "$or": [
        { "qty": { "$gt": 5 }},
        { "color": "yellow" }
    ]
})
list(results)

[{'_id': ObjectId('684369047bbf6b41012e9c0c'),
  'name': 'bananas',
  'qty': 7,
  'rating': 4,
  'color': 'yellow',
  'type': ['cavendish']},
 {'_id': ObjectId('684369047bbf6b41012e9c0d'),
  'name': 'oranges',
  'qty': 6,
  'rating': 2,
  'type': ['naval', 'mandarin']},
 {'_id': ObjectId('684369047bbf6b41012e9c0e'),
  'name': 'pineapple',
  'qty': 3,
  'rating': 5,
  'color': 'yellow'}]

## Otros

Ver [array operators](https://www.mongodb.com/docs/languages/python/pymongo-driver/current/read/specify-a-query/#array-operators), [element operators](https://www.mongodb.com/docs/languages/python/pymongo-driver/current/read/specify-a-query/#element-operators), [evaluation operators](https://www.mongodb.com/docs/languages/python/pymongo-driver/current/read/specify-a-query/#evaluation-operators)

- `$exists`

In [29]:
results = collection.find({ 
    "type": {
        "$exists": False
    }
})
list(results)

[{'_id': ObjectId('684369047bbf6b41012e9c0e'),
  'name': 'pineapple',
  'qty': 3,
  'rating': 5,
  'color': 'yellow'}]

# Proyección

In [30]:
results = collection.find({
    "type": {
        "$exists": True
    }
}, {"name": 1})
list(results)

[{'_id': ObjectId('684369047bbf6b41012e9c0b'), 'name': 'apples'},
 {'_id': ObjectId('684369047bbf6b41012e9c0c'), 'name': 'bananas'},
 {'_id': ObjectId('684369047bbf6b41012e9c0d'), 'name': 'oranges'}]

In [31]:
results = collection.find({
    "type": {
        "$exists": True
    }
}, {"name": 1, "_id": 0})
list(results)

[{'name': 'apples'}, {'name': 'bananas'}, {'name': 'oranges'}]

# Documentos a retornar

In [32]:
results = collection.find({
    "type": {
        "$exists": True
    }
}, {"name": 1, "rating": 1, "_id": 0}).sort("rating", pymongo.DESCENDING).skip(1).limit(2)
list(results)

[{'name': 'apples', 'rating': 3}, {'name': 'oranges', 'rating': 2}]

## Distinct

In [33]:
results = collection.distinct("color", {"rating": {"$gte": 4}})
list(results)

['yellow']

# Agregaciones

Es un data pipeline.

In [5]:
import requests
import bson.json_util as jsutil
import json

accounts_url = "https://raw.githubusercontent.com/neelabalan/mongodb-sample-dataset/main/sample_analytics/accounts.json"
customers_url = "https://raw.githubusercontent.com/neelabalan/mongodb-sample-dataset/main/sample_analytics/customers.json"
transactions_url = "https://raw.githubusercontent.com/neelabalan/mongodb-sample-dataset/main/sample_analytics/transactions.json"

In [6]:
jsutil.loads(requests.get(accounts_url).content.decode().split("\n")[0])

{'_id': ObjectId('5ca4bbc7a2dd94ee5816238c'),
 'account_id': 371138,
 'limit': 9000,
 'products': ['Derivatives', 'InvestmentStock']}

In [7]:
from tqdm.autonotebook import tqdm

db = client["analytics"]

db["accounts"].drop()
accounts = db["accounts"]
for line in tqdm(requests.get(accounts_url).content.decode().strip().split("\n")):
    accounts.insert_one(jsutil.loads(line))

db["customers"].drop()
customers = db["customers"]
for line in tqdm(requests.get(customers_url).content.decode().strip().split("\n")):
    customers.insert_one(jsutil.loads(line))    

db["transactions"].drop()
transactions = db["transactions"]
for line in tqdm(requests.get(transactions_url).content.decode().strip().split("\n")):
    transactions.insert_one(jsutil.loads(line))    

100%|██████████| 1746/1746 [00:00<00:00, 1906.33it/s]


In [8]:
pipeline = [
    { "$unwind": "$transactions" },
    { "$match": {"transactions.transaction_code": "buy", "transactions.symbol": {"$in": ["aapl", "msft", "nvda"]} }},
    {'$set': {'transactions.total': {'$toDouble': '$transactions.total'}}},
    { "$group": {"_id": "$transactions.symbol",
                 "operations": {"$sum": 1},
                 "volume": {"$sum": "$transactions.total"},
                 "shares_volume": {"$sum": "$transactions.amount"},
                "first_purchase": {"$first": "$transactions.date"}
                }},
    {"$sort": {"volume": 1}}
]

result = transactions.aggregate(pipeline)
list(result)

[{'_id': 'nvda',
  'operations': 2608,
  'volume': 237490543.13639304,
  'shares_volume': 13004534,
  'first_purchase': datetime.datetime(2011, 8, 23, 0, 0)},
 {'_id': 'msft',
  'operations': 2379,
  'volume': 272206501.04825485,
  'shares_volume': 11920398,
  'first_purchase': datetime.datetime(2002, 12, 4, 0, 0)},
 {'_id': 'aapl',
  'operations': 2444,
  'volume': 337539453.6043124,
  'shares_volume': 12123031,
  'first_purchase': datetime.datetime(1993, 9, 1, 0, 0)}]

In [9]:
customers.find_one({})

{'_id': ObjectId('5ca4bbcea2dd94ee58162a68'),
 'username': 'fmiller',
 'name': 'Elizabeth Ray',
 'address': '9286 Bethany Glens\nVasqueztown, CO 22939',
 'birthdate': datetime.datetime(1977, 3, 2, 2, 20, 31),
 'email': 'arroyocolton@gmail.com',
 'active': True,
 'accounts': [371138, 324287, 276528, 332179, 422649, 387979],
 'tier_and_details': {'0df078f33aa74a2e9696e0520c1a828a': {'tier': 'Bronze',
   'id': '0df078f33aa74a2e9696e0520c1a828a',
   'active': True,
   'benefits': ['sports tickets']},
  '699456451cc24f028d2aa99d7534c219': {'tier': 'Bronze',
   'benefits': ['24 hour dedicated line', 'concierge services'],
   'active': True,
   'id': '699456451cc24f028d2aa99d7534c219'}}}

Ver [la referencia](https://www.mongodb.com/docs/manual/aggregation/).

# Ejercicios
1. Leer un elemento cualquiera de cada una de las tres colecciones
2. Calcular cuantas cuentas tienen cada tipo de producto y el limite promedio que tienen
3. El [siguiente código](https://stackoverflow.com/a/60352517) es un template de como crear [una vista](https://www.mongodb.com/docs/manual/core/views/join-collections-with-view/):

```python
db.create_collection(
    'parsed_tests_view',
    viewOn='parsed_tests',
    pipeline=[{
        '$lookup': {
            'from': "raw_tests",
            'localField': "repository_path",
            'foreignField': "repository_path",
            'as': "raw_data"
        }
    }]
)
```

Cree una vista uniendo `customers` con `accounts`.

4. Con la vista creada en el punto 3, muestre los mails de los 10 clientes con mayor limite de cuenta total sumando los limites de todas sus cuentas.
5. Arme un pipeline para calcular el precio promedio de cada acción

# Ejercicios EXTRA 
1. Contar el número total de transacciones por transaction_code (ej. 'buy', 'sell').
2. Obtener el número total de transacciones por account_id para cuentas con al menos 50 transacciones.
3. Encontrar el promedio del limit de las cuentas activas que ofrecen 'InvestmentStock'.
4. Obtener el número de clientes que tienen acceso a cada tipo de producto (ej. Derivatives, InvestmentStock), solo para clientes activos.